In [ ]:

from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd
import plotly.express as px

In [ ]:
def query_yakir_experiment_data(client, dataset_id, table_id):
    """
    Query to get experiment data for Yakir with the number of entries, first timestamp, last timestamp, 
    and unique sensors.

    Parameters:
        client (bigquery.Client): Initialized BigQuery client.
        dataset_id (str): The ID of the dataset.
        table_id (str): The ID of the table.

    Returns:
        pd.DataFrame: DataFrame containing the query results.
    """
    query = f"""
    SELECT
        YakirGROUP.ExperimentData_Exp_name,
        COUNT(*) AS num_entries,
        MIN(YakirGROUP.TimeStamp) AS first_timestamp,
        MAX(YakirGROUP.TimeStamp) AS last_timestamp,
        ARRAY_AGG(DISTINCT YakirGROUP.SensorData_Name) AS unique_sensors
    FROM
        `iucc-f4d.{dataset_id}.{table_id}` AS YakirGROUP
    GROUP BY
        YakirGROUP.ExperimentData_Exp_name
    ORDER BY
        SAFE_CAST(REGEXP_EXTRACT(YakirGROUP.ExperimentData_Exp_name, r'exp_(\\d+)') AS INT64);
    """

    query_job = client.query(query)
    results = query_job.result()

    # Convert query results to a DataFrame
    data = []
    for row in results:
        data.append({
            "ExperimentData_Exp_name": row.ExperimentData_Exp_name,
            "num_entries": row.num_entries,
            "first_timestamp": row.first_timestamp,
            "last_timestamp": row.last_timestamp
        })

    return pd.DataFrame(data)



In [ ]:
def query_sensor_data(client, dataset_id, table_id, experiment_name, start_time, end_time):
    """
    Query sensor data for a specific experiment within a given time range.

    Parameters:
        client (bigquery.Client): Initialized BigQuery client.
        dataset_id (str): The ID of the dataset.
        table_id (str): The ID of the table.
        experiment_name (str): The name of the experiment to filter.
        start_time (str): The start of the time range (inclusive).
        end_time (str): The end of the time range (inclusive).

    Returns:
        pd.DataFrame: DataFrame containing the query results.
    """
    query = f"""
    SELECT
        Yakir.TimeStamp,
        Yakir.SensorData_Name,
        Yakir.SensorData_humidity,
        Yakir.SensorData_temperature,
        Yakir.SensorData_light,
        Yakir.SensorData_barometric_pressure,
    FROM
        `iucc-f4d.{dataset_id}.{table_id}` AS Yakir
    WHERE
        Yakir.ExperimentData_Exp_name = '{experiment_name}'
        AND Yakir.TimeStamp BETWEEN '{start_time}' AND '{end_time}';
    """

    query_job = client.query(query)
    results = query_job.result()

    # Convert query results to a DataFrame
    data = []
    for row in results:
        data.append({
            "TimeStamp": row.TimeStamp,
            "SensorData_Name": row.SensorData_Name,
            "SensorData_humidity": row.SensorData_humidity,
            "SensorData_temperature": row.SensorData_temperature,
            "SensorData_light": row.SensorData_light,
            "SensorData_barometric_pressure": row.SensorData_barometric_pressure
        })
    df = pd.DataFrame(data)
    df.sort_values(by='TimeStamp', inplace=True)
    return df

# Here you read and connect to the GCP server

In [ ]:

# Path to your credentials JSON file
credentials_path = "read_BQ.json"

# Load credentials from the JSON file
credentials = service_account.Credentials.from_service_account_file(credentials_path)

# Initialize BigQuery client
client = bigquery.Client(credentials=credentials, project="iucc-f4d")

# Initialize an empty list to store data for the DataFrame
data = []

dataset_costs = {}  # Dictionary to store costs per dataset
combined_cost = 0   # Variable to track the combined cost

In [ ]:
dataset_id = "Yakir"
table_id = "d83adde260d1"
df_experiment = query_yakir_experiment_data(client, dataset_id, table_id)
df_experiment

,ExperimentData_Exp_name,num_entries,first_timestamp,last_timestamp
0,exp_3_Check_up_2,713,2025-01-05 12:18:00+00:00,2025-01-05 13:58:00+00:00
1,exp_4_new_cover,12,2025-01-05 15:23:00+00:00,2025-01-05 15:32:00+00:00
2,exp_5_new_cover_2,15,2025-01-05 15:50:00+00:00,2025-01-05 16:02:00+00:00
3,exp_6_new_cover,7206,2025-01-05 16:11:00+00:00,2025-01-09 10:47:00+00:00
4,exp_7_test_all_together,21218,2025-01-14 14:47:00+00:00,2025-01-16 12:56:00+00:00
5,exp_8_calib_temp,5995,2025-01-19 17:26:00+00:00,2025-01-22 08:19:00+00:00
6,exp_9_Yatir_1,48767,2025-01-22 11:09:00+00:00,2025-02-12 12:12:00+00:00
7,exp_10_Yatir_all,275166,2025-02-12 14:33:00+00:00,2025-04-03 18:29:00+00:00
8,exp_11_check,18,2025-03-16 12:29:00+00:00,2025-03-16 13:20:00+00:00


In [ ]:
experiment_name = "exp_10_Yatir_all"
start_time ,end_time = "2025-03-01", "2025-03-04"

df_data = query_sensor_data(client, dataset_id, table_id, experiment_name, start_time, end_time)
# print the estimated cost of the query

In [ ]:
# Define the parameters for the plot
# 'SensorData_humidity', 'SensorData_temperature', 'SensorData_light','
y_axis = 'SensorData_barometric_pressure'  # Choose the y-axis parameter

# Generate the line plot based on the selected parameter
fig = px.line(df_data, x='TimeStamp', y=y_axis, color='SensorData_Name', title=f'{y_axis} Over Time')
fig.show()
